In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

ImportError: cannot import name 'display' from 'IPython.core.display' (/opt/anaconda3/envs/ironhack/lib/python3.12/site-packages/IPython/core/display.py)

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [4]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [20]:
# Your code
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(
    data, 
    test_size=0.2, 
    random_state=42
)

## Data Preprocessing

In [6]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [7]:
# Your code
import re

def clean_html(html_content):
    html_content = re.sub(r'<(script|style).*?>.*?</\1>', '', html_content, flags=re.DOTALL | re.IGNORECASE)
    html_content = re.sub(r'', '', html_content, flags=re.DOTALL)
    clean_text = re.sub(r'<.*?>', '', html_content)
    clean_text = " ".join(clean_text.split())
    return clean_text

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [8]:
# Your code
def final_clean(text):
    # 1. Remove all the special characters (keep only letters and spaces)
    text = re.sub(r'\W', ' ', text)
    
    # 2. Remove numbers
    text = re.sub(r'\d', ' ', text)
    
    # 3. Remove all single characters (words like 'a' or 'i' surrounded by spaces)
    text = re.sub(r'\s+[a-zA-Z]\s+', ' ', text)
    
    # 4. Remove single characters from the start of the string
    text = re.sub(r'^[a-zA-Z]\s+', '', text)
    
    # 5. Substitute multiple spaces with single space
    text = re.sub(r'\s+', ' ', text, flags=re.I)
    
    # 6. Remove prefixed 'b' 
    text = re.sub(r'^b\s+', '', text)
    
    # 7. Convert to Lowercase
    text = text.lower()
    
    return text.strip()

## Now let's work on removing stopwords
Remove the stopwords.

In [9]:
# Your code
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return " ".join(filtered_words)

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/anastasiakabanovsky/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [10]:
# Your code
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = text.split()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized_words)

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/anastasiakabanovsky/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [25]:
# Your code
from collections import Counter

data_train['text'] = data_train['text'].apply(clean_html).apply(final_clean).apply(remove_stopwords).apply(lemmatize_text)
data_val['text'] = data_val['text'].apply(clean_html).apply(final_clean).apply(remove_stopwords).apply(lemmatize_text)
ham = data_train[data_train['label'] == 0]['text']
spam = data_train[data_train['label'] == 1]['text']

all_text_ham = " ".join(ham).split(" ")
all_text_spam = " ".join(spam).split(" ")

ham_most_common = Counter(all_text_ham).most_common(10)
spam_most_common = Counter(all_text_spam).most_common(10)

## Extra features

In [24]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,money_mark,suspicious_words,text_len
29,"----------- REGARDS, MR NELSON SMITH.KINDLY RE...",1,0,0,106
535,I have not been able to reach oscar this am. W...,0,0,0,101
695,; Huma Abedin B6I'm checking with Pat on the 5...,0,0,0,141
557,I can have it announced here on Monday - can't...,0,0,0,52
836,BANK OF AFRICAAGENCE SAN PEDRO14 BP 1210 S...,1,0,1,1750


## How would work the Bag of Words with Count Vectorizer concept?

In [27]:
# Your code
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X_train_counts = vectorizer.fit_transform(data_train['text'])
print(X_train_counts[0].toarray())

[[0 0 0 ... 0 0 0]]


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [28]:
# Your code
vectorizer = TfidfVectorizer()
tfidf = vectorizer.fit_transform(data_train['text'])
print(tfidf.shape)

(800, 28297)


## And the Train a Classifier?

In [29]:
# Your code
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Fit the model 
model = LogisticRegression()
model.fit(tfidf, data_train['label'])

# Prepare the data 
tfidf_test = vectorizer.transform(data_val['text'])

# Make predictions
predictions = model.predict(tfidf_test)

# Evaluate the model 
print(f"Accuracy: {accuracy_score(data_val['label'], predictions)}")
print(classification_report(data_val['label'], predictions))

Accuracy: 0.96
              precision    recall  f1-score   support

           0       0.94      1.00      0.97       125
           1       1.00      0.89      0.94        75

    accuracy                           0.96       200
   macro avg       0.97      0.95      0.96       200
weighted avg       0.96      0.96      0.96       200



### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code